# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports

import os
import time
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# OpenAI client for gpt-4o-mini
openai = OpenAI()

# Ollama exposes an OpenAI-compatible endpoint, so the same client class works locally.
# (Make sure Ollama is running and you have pulled the model: ollama pull llama3.2)
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

system_prompt = """You are a patient senior engineer and technical tutor.
When given a technical question or code snippet, explain it clearly:
1. A one-line summary of what it does.
2. A step-by-step walkthrough.
3. Why someone would write it this way, and a simpler or more explicit alternative.
Use markdown, and put code in fenced code blocks."""

def build_messages(question):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming
def stream_answer(client, model, question):
    """Stream a model's answer as live markdown. Returns (full_text, seconds)."""
    start = time.time()
    stream = client.chat.completions.create(
        model=model,
        messages=build_messages(question),
        stream=True,
    )
    response = ""
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=handle.display_id)
    return response, time.time() - start

gpt_answer, gpt_seconds = stream_answer(openai, MODEL_GPT, question)
print(f"\n{MODEL_GPT} took {gpt_seconds:.1f}s")

In [ ]:
# Get Llama 3.2 to answer

llama_answer, llama_seconds = stream_answer(ollama, MODEL_LLAMA, question)
print(f"\n{MODEL_LLAMA} took {llama_seconds:.1f}s")

## Bonus: a reusable tutor tool

`ask_both(question)` sends the same question to GPT and Llama one after the other and prints a small timing table, so I can use this tool during the rest of the course and compare cloud vs local models.

In [ ]:
def ask_both(question):
    display(Markdown(f"### {MODEL_GPT} (cloud)"))
    _, gpt_s = stream_answer(openai, MODEL_GPT, question)
    display(Markdown(f"### {MODEL_LLAMA} (local, Ollama)"))
    try:
        _, llama_s = stream_answer(ollama, MODEL_LLAMA, question)
    except Exception as e:
        print(f"Could not reach Ollama - is it running? ({e})")
        llama_s = None
    rows = f"| Model | Seconds |\n|---|---|\n| {MODEL_GPT} | {gpt_s:.1f} |\n"
    if llama_s is not None:
        rows += f"| {MODEL_LLAMA} | {llama_s:.1f} |\n"
    display(Markdown(rows))

ask_both("What is the difference between a list comprehension and a generator expression in Python?")